# Manual validation of the predicted endothelial cell states

**Developed by:** Anna Maguza\
**Affiliation:** Faculty of Medicine, Würzburg University\
**Creation date:** 28th February 2024\
**Last modified date:** 28th February 2024

This notebook outlines the process of validation of predicted by `scVI - scANVI` pipeline endothelial cell states annotations. We will evaluate `scANVI` classificator confidence and also validate the predicted annotation with markers.

## Import packages

In [1]:
import numpy as np
import scanpy as sc
import anndata
import pandas as pd
import plotnine as p
import matplotlib.pyplot as plt
import seaborn as sns

import json
from datetime import datetime

## Setup working environment

In [2]:
sc.settings.verbosity = 3
sc.settings.set_figure_params(dpi = 180, color_map = 'magma_r', dpi_save = 300, vector_friendly = True, format = 'svg')

In [3]:
timestamp = datetime.now().strftime('%d%m%Y_%H%M%S')

In [ ]:
timestamp

## Upload data

In [ ]:
adata = sc.read_h5ad('data/gut_data/gut_hs_all_datasets_scVI_scANVI_endothelial_cellstates_AM_28022025_102008_raw.h5ad')
adata

## Validate confidence score and markers expression

In [ ]:
adata_log = adata.copy()
sc.pp.normalize_total(adata_log, target_sum=1e6, exclude_highly_expressed=True)
sc.pp.log1p(adata_log)

In [ ]:
with plt.rc_context():
    sc.set_figure_params(dpi=300, figsize=(15, 12))
    sc.pl.umap(adata_log,color=["cellstates_scANVI", "confidence_score"], ncols=1, color_map = 'magma_r', frameon=False, show=False, size = 10)
    plt.savefig(f"/Users/annamaguza/Desktop/Repos/Fetal_stem_cells_analysis/7_endothelial_cell_states_annotation/figures/endothelial_scANVI_confidence_{timestamp}.png", bbox_inches="tight")
    plt.show()

## Validate markers expression

In [27]:
def visualize_cell_state_markers(adata, cell_state, markers, timestamp=None):
    """
    Create temporary annotation for specific cell state and visualize with markers.
    
    Parameters:
    -----------
    adata : AnnData
        Annotated data matrix
    cell_state : str
        Cell state of interest from cellstates_scANVI column
    markers : list
        List of marker genes to visualize
    timestamp : str, optional
        Timestamp for file naming. If None, current time will be used
    """

    if timestamp is None:
        timestamp = datetime.now().strftime('%d%m%Y_%H%M%S')

    # Create temporary column for cell state
    temp_col_name = 'temp_cell_state'
    adata.obs[temp_col_name] = adata.obs['cellstates_scANVI'] == cell_state
    
    # Set up colors in uns
    original_colors = None
    if 'temp_cell_state_colors' in adata.uns:
        original_colors = adata.uns['temp_cell_state_colors'].copy()
    
    # Set up color palette
    adata.uns[f'{temp_col_name}_colors'] = ['#D3D3D3', '#ba0f30']  # Light pink for target, light grey for others
    
    # Calculate plot layout
    n_total_plots = len(markers) + 1  # +1 for cell state plot
    n_cols = 3
    n_rows = (n_total_plots + (n_cols - 1)) // n_cols  # Ceiling division
    
    # Create the plot
    with plt.rc_context():
        sc.set_figure_params(dpi=300, figsize=(15, 5*n_rows))
        fig, axes = plt.subplots(n_rows, n_cols, figsize=(15, 5*n_rows))
        
        # Convert axes to 1D array if necessary
        if n_rows == 1:
            axes = np.array([axes])  # Handle single row case
        axes = axes.flatten()
        
        # Plot cell state
        sc.pl.umap(adata, color=[temp_col_name], ax=axes[0], 
                  frameon=False, title=f'{cell_state} cells',
                  show=False, size=7)
        
        # Plot markers
        for i, marker in enumerate(markers):
            if i + 1 >= len(axes):
                break
            if marker in adata.var_names:
                sc.pl.umap(adata, color=marker, ax=axes[i + 1],
                          color_map='magma_r', frameon=False,
                          title=marker, show=False, size=3)
            else:
                print(f"Warning: Marker {marker} not found in the data")
                axes[i + 1].set_title(f"{marker} (not found)")
                axes[i + 1].axis('off')
        
        # Remove empty subplots
        for i in range(len(markers) + 1, len(axes)):
            fig.delaxes(axes[i])
        
        plt.tight_layout()
        # Save figure
        plt.savefig(
            f"figures/{cell_state}_markers_{timestamp}.png", 
            bbox_inches="tight"
        )
        plt.show()
    
    # Clean up: remove temporary column and colors
    del adata.obs[temp_col_name]
    if f'{temp_col_name}_colors' in adata.uns:
        del adata.uns[f'{temp_col_name}_colors']
    
    # Restore original colors if they existed
    if original_colors is not None:
        adata.uns['temp_cell_state_colors'] = original_colors

In [ ]:
adata_log.obs['cellstates_scANVI'].value_counts()

+ Venous capillary

In [ ]:
markers = ['ACKR1', 'VWF', "RGCC", "VWA1"]
visualize_cell_state_markers(adata_log, "venous capillary", markers, timestamp)

+ Venous EC

In [ ]:
markers = ['ACKR1', 'VWF']
visualize_cell_state_markers(adata_log, "Venous EC", markers, timestamp)

* Arterial EC

In [ ]:
markers = ['GJA4', 'HEY1', 'HEY2', 'EFNB2']
visualize_cell_state_markers(adata_log, "Arterial EC", markers, timestamp)

* LEC

In [ ]:
markers = ["PROX1", "PDPN", "LYVE1", "FLT4", "CCL21", "COLEC12", "TBX1", "FOXC2"]
visualize_cell_state_markers(adata_log, "LEC", markers, timestamp)

* Cycling EC

In [ ]:
markers = ["MKI67", "TOP2A", 'UBE2C']
visualize_cell_state_markers(adata_log, "cycling EC", markers, timestamp)

* Arterial Capillary EC

In [ ]:
markers = ["CA4", "FCN3", "RGCC", "VWA1"]
visualize_cell_state_markers(adata_log, "arterial capillary", markers, timestamp)